# MySQL → BigQuery Sandbox Upload

Run each cell top to bottom — that's it.

**Before running this notebook, do this ONE TIME in your terminal:**
```
gcloud auth application-default login
```
This logs you into Google. After that, this notebook handles everything automatically.

In [1]:
# ── Install required packages ─────────────────────────────────────────────────
# Run this cell once. After packages are installed you can skip it next time.

import subprocess, sys

packages = [
    "google-cloud-bigquery",
    "google-cloud-bigquery-storage",
    "pyarrow",
    "pandas",
    "pymysql",
    "sqlalchemy",
]

for pkg in packages:
    print(f"Installing {pkg}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("\n✅ All packages installed.")

Installing google-cloud-bigquery...
Installing google-cloud-bigquery-storage...
Installing pyarrow...
Installing pandas...
Installing pymysql...
Installing sqlalchemy...

✅ All packages installed.


In [3]:
# ── CONFIG — only edit this cell ─────────────────────────────────────────────

# Your MySQL connection details
MYSQL_CONFIG = {
    "host":     "localhost",
    "user":     "root",
    "password": "11111",
    "database": "shipping_management",
    "port":     3306,
}

# Your BigQuery details
# Find your Project ID at: console.cloud.google.com → top-left dropdown
# It looks like:  my-project-123456
BQ_PROJECT_ID = "shipment-data-analysis189"   # ← replace this
BQ_DATASET_ID = "shipping_db"           # BigQuery dataset name (auto-created if missing)
BQ_LOCATION   = "US"                    # Change to 'EU' if you're in Europe

# All 17 tables to upload
TABLES = [
    "customers",
    "service_areas",
    "service_area_zip_codes",
    "employees",
    "warehouses",
    "vendors",
    "vehicles",
    "maintenance_records",
    "fuel_purchases",
    "shipments",
    "delivery_routes",
    "insurance_claims",
    "customer_service_tickets",
    "packages",
    "route_stops",
    "inventory",
    "invoices",
]

print("✅ Config ready.")

✅ Config ready.


In [5]:
# ── Imports ───────────────────────────────────────────────────────────────────

import pandas as pd
from sqlalchemy import create_engine
from google.cloud import bigquery

print("✅ Imports successful.")

✅ Imports successful.


In [7]:
# ── Test MySQL connection ─────────────────────────────────────────────────────

def get_mysql_engine():
    cfg = MYSQL_CONFIG
    url = (
        f"mysql+pymysql://{cfg['user']}:{cfg['password']}"
        f"@{cfg['host']}:{cfg['port']}/{cfg['database']}"
    )
    return create_engine(url, pool_pre_ping=True)

try:
    engine = get_mysql_engine()
    with engine.connect() as conn:
        result = pd.read_sql("SHOW TABLES", conn)
    print(f"✅ MySQL connected. Found {len(result)} tables:")
    print(result.to_string(index=False))
except Exception as e:
    print(f"❌ MySQL connection failed: {e}")

✅ MySQL connected. Found 17 tables:
Tables_in_shipping_management
     customer_service_tickets
                    customers
              delivery_routes
                    employees
               fuel_purchases
             insurance_claims
                    inventory
                     invoices
          maintenance_records
                     packages
                  route_stops
       service_area_zip_codes
                service_areas
                    shipments
                     vehicles
                      vendors
                   warehouses


In [13]:
# ── Test BigQuery connection ───────────────────────────────────────────────────
# This uses your  `gcloud auth application-default login`  credentials.
# No JSON key file needed.

try:
    bq_client = bigquery.Client(project=BQ_PROJECT_ID)
    # Simple test — list datasets in your project
    datasets = list(bq_client.list_datasets())
    print(f"✅ BigQuery connected to project: {BQ_PROJECT_ID}")
    if datasets:
        print(f"   Existing datasets: {[d.dataset_id for d in datasets]}")
    else:
        print("   No datasets yet — will create 'shipping_db' in the next cell.")
except Exception as e:
    print(f"❌ BigQuery connection failed: {e}")
    print()
    print("   Most likely fix: run this in your terminal and try again:")
    print("   gcloud auth application-default login")

✅ BigQuery connected to project: shipment-data-analysis189
   No datasets yet — will create 'shipping_db' in the next cell.


In [15]:
# ── Create BigQuery dataset if it doesn't exist ───────────────────────────────

dataset_ref = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}"

try:
    bq_client.get_dataset(dataset_ref)
    print(f"✅ Dataset '{BQ_DATASET_ID}' already exists.")
except Exception:
    dataset = bigquery.Dataset(dataset_ref)
    dataset.location = BQ_LOCATION
    bq_client.create_dataset(dataset)
    print(f"✅ Dataset '{BQ_DATASET_ID}' created successfully.")

✅ Dataset 'shipping_db' created successfully.


In [17]:
# ── Upload function ───────────────────────────────────────────────────────────
# This function handles one table at a time:
#   1. Read from MySQL
#   2. Fix column types that BigQuery doesn't like
#   3. Upload to BigQuery (replaces table if it already exists)

def upload_table(table_name):
    print(f"\n[{table_name}] Starting...")

    # Step 1: Read from MySQL
    engine = get_mysql_engine()
    df = pd.read_sql(f"SELECT * FROM `{table_name}`", con=engine)
    print(f"[{table_name}] Read {len(df):,} rows from MySQL.")

    if df.empty:
        print(f"[{table_name}] Table is empty — skipping.")
        return 0

    # Step 2: Fix types
    # MySQL TIME columns become Python timedelta — BigQuery can't handle those.
    # Convert them to strings like '08:00:00'.
    for col in df.columns:
        if pd.api.types.is_timedelta64_dtype(df[col]):
            df[col] = df[col].apply(
                lambda x: str(x).split("days ")[-1].strip() if pd.notna(x) else None
            )

    # Step 3: Upload to BigQuery
    destination = f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}.{table_name}"

    job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,  # safe to re-run
        autodetect=True,  # BigQuery auto-detects column types
    )

    job = bq_client.load_table_from_dataframe(df, destination, job_config=job_config)
    job.result()  # wait for upload to finish

    bq_table = bq_client.get_table(destination)
    print(f"[{table_name}] ✅ Done — {bq_table.num_rows:,} rows in BigQuery.")
    return bq_table.num_rows

print("✅ Upload function defined.")

✅ Upload function defined.


In [19]:
# ── Upload all 17 tables ──────────────────────────────────────────────────────
# This cell is the main upload. It will take several minutes depending on your
# internet speed (you're uploading ~350k rows total).
#
# Progress is printed for each table.
# If one table fails, the rest still continue.

results = {}   # table_name -> row count or error message

for table in TABLES:
    try:
        row_count = upload_table(table)
        results[table] = f"✅  {row_count:,} rows"
    except Exception as e:
        results[table] = f"❌  ERROR: {e}"
        print(f"[{table}] ❌ Failed: {e}")

# Print summary
print("\n" + "="*55)
print("  UPLOAD SUMMARY")
print("="*55)
for table, status in results.items():
    print(f"  {table:<35} {status}")

success_count = sum(1 for s in results.values() if s.startswith("✅"))
print("="*55)
print(f"  {success_count}/{len(TABLES)} tables uploaded successfully.")


[customers] Starting...
[customers] Read 5,001 rows from MySQL.


C:\Users\nitin\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[customers] ✅ Done — 5,001 rows in BigQuery.

[service_areas] Starting...
[service_areas] Read 101 rows from MySQL.


C:\Users\nitin\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[service_areas] ✅ Done — 101 rows in BigQuery.

[service_area_zip_codes] Starting...
[service_area_zip_codes] Read 1,011 rows from MySQL.


...

[route_stops] ✅ Done — 30,000 rows in BigQuery.

[inventory] Starting...
[inventory] Read 100,000 rows from MySQL.


C:\Users\nitin\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[inventory] ✅ Done — 100,000 rows in BigQuery.

[invoices] Starting...
[invoices] Read 50,001 rows from MySQL.


C:\Users\nitin\anaconda3\Lib\site-packages\google\cloud\bigquery\_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


[invoices] ✅ Done — 50,001 rows in BigQuery.

  UPLOAD SUMMARY
  customers                           ✅  5,001 rows
  service_areas                       ✅  101 rows
  service_area_zip_codes              ✅  1,011 rows
  employees                           ✅  1,001 rows
  warehouses                          ✅  51 rows
  vendors                             ✅  201 rows
  vehicles                            ✅  501 rows
  maintenance_records                 ✅  3,001 rows
  fuel_purchases                      ✅  5,001 rows
  shipments                           ✅  50,001 rows
  delivery_routes                     ✅  3,001 rows
  insurance_claims                    ✅  1,001 rows
  customer_service_tickets            ✅  1,501 rows
  packages                            ✅  100,001 rows
  route_stops                         ✅  30,000 rows
  inventory                           ✅  100,000 rows
  invoices                            ✅  50,001 rows
  17/17 tables uploaded successfully.


In [28]:
# ── Verify — check all tables are in BigQuery ────────────────────────────────
# Uses list_tables() + get_table() instead of __TABLES__ query.
# This approach works reliably on sandbox BigQuery.

print(f"Tables in BigQuery dataset '{BQ_DATASET_ID}':") 
print()

rows = []
for bq_table_item in bq_client.list_tables(f"{BQ_PROJECT_ID}.{BQ_DATASET_ID}"):
    # get_table() fetches full metadata including row count and byte size
    t = bq_client.get_table(bq_table_item.reference)
    size_mb = round(t.num_bytes / 1024 / 1024, 2) if t.num_bytes else 0.0
    rows.append({
        "table":   t.table_id,
        "rows":    t.num_rows,
        "size_mb": size_mb,
    })

summary_df = pd.DataFrame(rows).sort_values("rows", ascending=False).reset_index(drop=True)

print(summary_df.to_string(index=False))
print()
print(f"Total tables : {len(summary_df)}")
print(f"Total rows   : {summary_df['rows'].sum():,}")
print(f"Total size   : {summary_df['size_mb'].sum():.2f} MB")


Tables in BigQuery dataset 'shipping_db':

                   table   rows  size_mb
                packages 100001    17.07
               inventory 100000     7.44
               shipments  50001     9.86
                invoices  50001     4.96
             route_stops  30000     3.96
          fuel_purchases   5001     0.47
               customers   5001     0.79
     maintenance_records   3001     0.48
         delivery_routes   3001     0.31
customer_service_tickets   1501     0.21
  service_area_zip_codes   1011     0.01
        insurance_claims   1001     0.13
               employees   1001     0.17
                vehicles    501     0.07
                 vendors    201     0.02
           service_areas    101     0.01
              warehouses     51     0.01

Total tables : 17
Total rows   : 351,375
Total size   : 45.97 MB
